In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import numpy as np

In [2]:
raw = pd.read_csv("../data/gate_orders.csv")
customers = pd.read_csv("../data/gate_customers.csv")

In [3]:
raw.shape
raw.dtypes
raw.isna().sum()
raw['product_category'].nunique() #problematic because the same categories are in diff cases returning a larger nunique

(420, 9)

order_id                str
customer_id             str
order_date              str
product_category        str
quantity              int64
unit_price              str
discount_code           str
status                  str
rating              float64
dtype: object

order_id              0
customer_id           0
order_date            0
product_category      0
quantity              0
unit_price            0
discount_code       292
status                0
rating              176
dtype: int64

17

In [4]:
def load_orders(path):
    return(
        pd.read_csv(path, parse_dates=["order_date"])
        .assign(category= lambda d: d['product_category'].str.strip().str.lower(),
                price= lambda d: d['unit_price'].str.replace('\$', "", regex=True).astype(float),
                revenue= lambda d: d['quantity']*d['price'])
        .drop(columns= ['product_category', 'unit_price'])
    )           

In [5]:
orders=load_orders('../data/gate_orders.csv')
orders['category'].nunique()
orders['category'].value_counts()
orders['revenue'].sum().round(2)

5

category
home           114
electronics    100
beauty          81
toys            68
books           57
Name: count, dtype: int64

np.float64(51164.04)

In [6]:
start= "2026-03-01"
end= "2026-03-31"
len(orders
.query('status== "delivered" and @start <= order_date <= @end and revenue>100')  
)

18

In [7]:
delivered= orders[orders['status']=='delivered']
category_report=delivered.groupby('category').agg(
    n_orders= ('order_id', 'count'),
    total_revenue= ('revenue', 'sum'),
    avg_rating= ('rating', 'mean')
).round(2).sort_values('total_revenue', ascending= False)
category_report
#home has maximum orders but electronics produce most revenue because the price of electronics would be higher returning a higher revenue on lower orders.

,n_orders,total_revenue,avg_rating
category,,,
electronics,82,23989.30,2.78
home,96,9940.62,2.91
toys,53,3787.24,2.75
beauty,65,3101.77,3.09
books,44,1276.42,2.77


In [8]:
orders['cat_avg']= orders.groupby('category')['revenue'].transform('mean')
(orders['revenue']>orders['cat_avg']).sum()

np.int64(160)

In [9]:
orders['rating'].isna().mean().round(3)
((orders['status'] != 'delivered') & orders['rating'].isna()).sum()
(orders['status'] != 'delivered').sum()
#a missing rating is because the order is not delivered. These should not be filled.
delivered= orders[orders['status']=='delivered']
delivered['rating_was_missing']= delivered['rating'].isna()
delivered['rating_filled']= delivered['rating'].fillna(delivered.groupby('category')['rating'].transform('median'))
delivered['rating'].mean().round(2), delivered['rating_filled'].mean().round(2)

np.float64(0.419)

np.int64(80)

np.int64(80)

(np.float64(2.87), np.float64(2.88))

In [10]:
customers['customer_id'].nunique()==len(customers)
merged= orders.merge(customers, on='customer_id', how='left')
orders.shape, customers.shape, merged.shape
(merged['tier'].isna()).sum()
merged.loc[merged['tier'].isna(), 'customer_id'].unique()
set(customers['customer_id'].unique())- set(orders['customer_id'].unique())

True

((420, 11), (120, 4), (420, 14))

np.int64(10)

<ArrowStringArray>
['C123', 'C121', 'C122']
Length: 3, dtype: str

{'C010', 'C068'}

In [11]:
merged.pivot_table(index= 'tier', columns= 'category', values= 'revenue', aggfunc= 'sum', fill_value=0).round(0)
#The comment misses the real reason: a tier/category combination with no orders genuinely made £0, so 0 is a true answer for a sum.

category,beauty,books,electronics,home,toys
tier,,,,,
basic,2131.0,910.0,12572.0,5177.0,1947.0
plus,929.0,276.0,9911.0,4204.0,943.0
premium,648.0,517.0,5794.0,2032.0,1696.0


In [12]:
monthly_rev = merged.set_index('order_date').resample('MS')['revenue'].sum()
monthly_rev.idxmax()
(merged['order_date'].dt.dayofweek>= 5).mean().round(3)

Timestamp('2026-05-01 00:00:00')

np.float64(0.329)

In [13]:
pd.crosstab(merged['tier'], merged['status'], normalize='index').round(3)
#basic has highest returns
merged['tier']= pd.Categorical(merged['tier'], categories=['basic', 'plus', 'premium'], ordered=True)
(merged['tier']>='plus').sum()
#The old table (status as rows) showed tier share within each status

status,cancelled,delivered,returned
tier,,,
basic,0.078,0.804,0.118
plus,0.120,0.786,0.094
premium,0.056,0.876,0.067


np.int64(206)